## Load clean train and test data

In [1]:
import pandas as pd

train_clean_dummies = pd.read_csv('train_test_clean/train_clean_dummies.csv')
train_clean_no_dummies = pd.read_csv('train_test_clean/train_clean_no_dummies.csv')

test_clean_dummies = pd.read_csv('train_test_clean/test_clean_dummies.csv')
test_clean_no_dummies = pd.read_csv('train_test_clean/test_clean_no_dummies.csv')

## Get time series split for training data, dummies / no dummies

In [2]:
from scripts.time_series_split import get_moving_windows_split_train, print_folds_shape

target_years = [2015, 2016, 2017, 2018]

train_dummies = get_moving_windows_split_train(train_clean_dummies, target_years, dummies='Y')
train = get_moving_windows_split_train(train_clean_no_dummies, target_years, dummies='N')

#print_folds_shape(train_dummies)
#print_folds_shape(train)

## Preprocess training data for models

In [3]:
from scripts.preprocess_for_models import transform_datatype_for_ebm, sanitize_column_names_for_xgb, apply_preprocessing_to_folds

train_ebm = apply_preprocessing_to_folds(train, transform_datatype_for_ebm)
train_xgb = apply_preprocessing_to_folds(train_dummies, sanitize_column_names_for_xgb)

## Hyperparameter tuning for XGBoost

In [ ]:
from scripts.hyperparameter_tuning import run_xgb_hyperparameter_tuning

# note: the format of training data may need to be adjusted, cf. preprocessing for xgb in main2

study = run_xgb_hyperparameter_tuning(train_dummies, n_trials=50)

## Hyperparameter tuning for EBM

In [ ]:
from scripts.hyperparameter_tuning import run_ebm_hyperparameter_tuning

# Optional: seed trial
initial_params = {
    'interactions': 20,
    'outer_bags': 10,
    'learning_rate': 0.01,
    'min_samples_leaf': 3,
    'max_leaves': 2
}

# Run only on first fold to save time
study_first = run_ebm_hyperparameter_tuning(train_ebm, n_trials=30, use_all_folds=False)

# Run on all folds; note that one trial takes about 30 minutes
study_all = run_ebm_hyperparameter_tuning(train_ebm, n_trials=30, use_all_folds=True)
